# 5장 오차역전파법 — 연습 문제

책 5장 순서(5.1 → 5.7)를 따른다. 해당 절을 읽고 나서 그 Part를 풀자.
`# 여기에 작성`을 채우고 실행(`Shift+Enter`)한 뒤 확인 셀에서 `통과!`가 나오면 성공.
막히면 `ch05_solutions.ipynb` 참고. 커널은 `.venv`.

**표시**
- 🔁 **배치 연습** — 책 범위 안이지만, 행렬·배치를 한꺼번에 다루는 연습으로 더 넣은 문제
- 🔮 **예측하기** — 실행하기 **전에** 답을 적고, 그다음 확인 셀로 맞춰 보기
- ⚠️ **함정** — 자주 틀리는 곳 (특히 **이름표와 상자**, `self.`, 딕셔너리)
- ⭐ **책 범위 밖** — 책 5장 본문에 없는 내용. 몰라도 진도에는 문제없음

**5장 한 줄 요약**: 4장의 수치 미분은 정확하지만 **느리다**. 5장은 계산 그래프와 **연쇄법칙**으로 기울기를 **한 번에** 구한다(오차역전파법).

## 0. 준비
아래 셀을 먼저 실행하자. 노트북을 다시 열었으면 **이 셀부터 다시 실행**하기.
4장에서 쓴 `common` 폴더의 함수를 그대로 쓴다.

In [ ]:
import sys, os
sys.path.append(os.pardir)   # 한 칸 위 폴더(C:\Deeplerning)의 dataset, common을 찾게 하기
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict
from common.functions import softmax, cross_entropy_error, sigmoid
from common.gradient import numerical_gradient   # 4장의 2차원용 수치 기울기

def numerical_diff(f, x):          # 4장의 수치 미분 (중심 차분)
    h = 1e-4
    return (f(x + h) - f(x - h)) / (2 * h)

# 채점 도우미 (먼저 실행해 두기)
def check(got, want, name='답'):
    if got is None:
        raise AssertionError(f'{name}: 아직 안 채웠어요 (None)')
    if isinstance(want, (tuple, str, bool)):
        ok = got == want
    elif isinstance(want, list) and want and isinstance(want[0], str):
        ok = list(got) == want
    else:
        ok = np.shape(got) == np.shape(want) and np.allclose(got, want)
    mark = '✅' if ok else '❌'
    print(f'{name} = {got!r}   (정답 {want!r}) {mark}')
    assert ok, f'{name} 이(가) 틀렸어요'

print('준비 완료')

---
# Part 1. 계산 그래프 (5.1)
## 1-1. 계산 그래프로 풀다 — 문제 1 (5.1.1)
> 슈퍼에서 1개에 100원인 사과를 2개 샀다. 소비세가 10%일 때 지불 금액은?

계산 그래프는 **왼쪽에서 오른쪽으로** 노드(○)를 따라 계산한다(**순전파**).
`100 → (×2) → (×1.1) → ?`
- `total1`: 지불 금액 (곱셈을 차례로)

In [ ]:
total1 = None   # 여기에 작성

In [ ]:
# 확인
check(total1, 220.0, 'total1')
print('통과!')

## 1-2. 문제 2 (5.1.1)
> 사과 2개(1개 100원), 귤 3개(1개 150원). 소비세 10%.

그래프: 사과 `100 ×2` → `apple_price`, 귤 `150 ×3` → `orange_price`, 둘을 **더하고**(+ 노드) → `all_price`, `×1.1` → `total2`

In [ ]:
apple_price = None    # 여기에 작성
orange_price = None   # 여기에 작성
all_price = None      # 여기에 작성
total2 = None         # 여기에 작성

In [ ]:
# 확인
check(apple_price, 200, 'apple_price'); check(orange_price, 450, 'orange_price')
check(all_price, 650, 'all_price'); check(total2, 715.0, 'total2')
print('통과!')

## 1-3. 국소적 계산 (5.1.2)
책 그림 5-4: 여러 식품을 사서 4,000원, 사과 2개 200원 → 더하기 노드가 4,200원을 낸다.
- `need_know`: 더하기 노드가 계산하려면 **4,000원이 어떻게(무엇을 사서) 나왔는지** 알아야 할까? (`True` / `False`)

> 각 노드는 **자기에게 들어온 값만** 보고 계산한다 = **국소적 계산**. 전체가 아무리 복잡해도 노드 하나는 단순하다.

In [ ]:
need_know = None   # 여기에 작성

In [ ]:
# 확인
check(need_know, False, 'need_know')
print('통과!')

## 1-4. 왜 계산 그래프로 푸는가? (5.1.3)
"**사과 가격이 오르면 최종 금액은 얼마나 오를까?**" = 사과 가격에 대한 지불 금액의 **미분**.
4장 방법(수치 미분)으로 먼저 구해 보자. **lambda 연습!**
- `total_of`: 사과 1개 가격 `apple`을 받아 지불 금액(`apple * 2 * 1.1`)을 돌려주는 **함수** (lambda로)
- `d_apple`: `numerical_diff(total_of, 100.0)`

> `total_of`에는 **함수**가 들어가야 한다. `total_of = 100 * 2 * 1.1`처럼 **숫자**를 넣으면 `numerical_diff` 안에서 `f(x + h)`를 부를 때 에러.

In [ ]:
total_of = None   # 여기에 작성 (lambda apple: ...)
d_apple = None    # 여기에 작성

In [ ]:
# 확인
check(d_apple, 2.2, 'd_apple')
print('통과! 사과가 1원 오르면 최종 금액은 2.2원 오른다.')
print('수치 미분은 "값을 살짝 바꿔 다시 계산"을 변수마다 해야 한다 → 변수가 많으면 느리다.')
print('역전파는 이 값을 계산 그래프를 거꾸로 한 번 지나가며 구한다 (Part 3에서 같은 2.2가 나온다).')

---
# Part 2. 연쇄법칙 (5.2)
## 2-1. 계산 그래프의 역전파 (5.2.1)
역전파 규칙: **상류(오른쪽)에서 온 값 × 그 노드의 국소적 미분** → 하류(왼쪽)로 전달.
- 상류에서 `1.5`가 왔고, 노드의 국소적 미분이 `4`라면 하류로 보내는 값 `down`은?

In [ ]:
down = None   # 여기에 작성

In [ ]:
# 확인
check(down, 6.0, 'down')
print('통과!')

## 2-2. 연쇄법칙 (5.2.2 ~ 5.2.3)
$z = t^2$, $t = x + y$ (합성 함수). $x = 1,\ y = 2$일 때 **손으로** 계산해 보자.
- `dz_dt`: $\partial z / \partial t = 2t$ 의 값
- `dt_dx`: $\partial t / \partial x$ 의 값
- `dz_dx`: 연쇄법칙 $\dfrac{\partial z}{\partial x} = \dfrac{\partial z}{\partial t}\dfrac{\partial t}{\partial x}$ 의 값

In [ ]:
dz_dt = None   # 여기에 작성
dt_dx = None   # 여기에 작성
dz_dx = None   # 여기에 작성

In [ ]:
# 확인 — 4장 수치 미분으로 검산 (y = 2로 고정하고 x만 움직이는 함수)
z_of_x = lambda x: (x + 2.0) ** 2
print('수치 미분:', numerical_diff(z_of_x, 1.0))
check(dz_dt, 6, 'dz_dt'); check(dt_dx, 1, 'dt_dx'); check(dz_dx, 6, 'dz_dx')
print('통과!')

## 2-3. 🔁 4장에서 내가 만든 예제를 연쇄법칙으로
4장에서 "왜 z가 아니라 W로 미분하나"를 이야기할 때 쓴 예제다.
$x = 2,\ w = 3,\ L = (xw - 4)^2$. 단계로 나누면 $a = xw$, $b = a - 4$, $L = b^2$

| 노드 | 순전파 | 국소적 미분 |
|:-:|:-:|:-:|
| `a = x * w` | 6 | $\partial a/\partial w = x$ |
| `b = a - 4` | 2 | $\partial b/\partial a = 1$ |
| `L = b ** 2` | 4 | $\partial L/\partial b = 2b$ |

- `dL_db`, `db_da`, `da_dw`: 각 국소적 미분의 **값**
- `dL_dw`: 셋을 곱한 값 (연쇄법칙)

In [ ]:
dL_db = None   # 여기에 작성
db_da = None   # 여기에 작성
da_dw = None   # 여기에 작성
dL_dw = None   # 여기에 작성

In [ ]:
# 확인
L_of_w = lambda w: (2.0 * w - 4) ** 2
print('수치 미분:', numerical_diff(L_of_w, 3.0))
check(dL_db, 4, 'dL_db'); check(db_da, 1, 'db_da'); check(da_dw, 2, 'da_dw'); check(dL_dw, 8, 'dL_dw')
print('통과! 4장에서 수치 미분으로 구한 8과 같다.')

---
# Part 3. 역전파 (5.3)
## 3-1. 덧셈 노드의 역전파 (5.3.1)
$z = x + y$ → $\partial z/\partial x = 1$, $\partial z/\partial y = 1$
상류에서 `dout = 1.3`이 왔다.
- `dx`, `dy`: 하류로 보내는 값

In [ ]:
dout = 1.3
dx = None   # 여기에 작성
dy = None   # 여기에 작성

In [ ]:
# 확인
check(dx, 1.3, 'dx'); check(dy, 1.3, 'dy')
print('통과! 덧셈 노드는 받은 값을 그대로 흘려보낸다.')

## 3-2. ⚠️ 곱셈 노드의 역전파 (5.3.2)
$z = xy$ → $\partial z/\partial x = y$, $\partial z/\partial y = x$ — **서로 바꿔서** 곱한다.
순전파 때 `x = 10`, `y = 5`였고 상류에서 `dout = 1.3`이 왔다.
- `dx`, `dy`

> 함정: `dx = dout * x`로 쓰기 쉽다. x 쪽으로 가는 값에는 **상대편(y)**을 곱한다.

In [ ]:
x, y = 10, 5
dout = 1.3
dx = None   # 여기에 작성
dy = None   # 여기에 작성

In [ ]:
# 확인
check(dx, 6.5, 'dx'); check(dy, 13.0, 'dy')
print('통과! 그래서 곱셈 노드는 순전파 때의 x, y를 기억해 둬야 한다 (Part 4의 self.x, self.y).')

## 3-3. 사과 쇼핑의 예 (5.3.3, 그림 5-14)
`apple(100) ─×─ apple_num(2)` → `apple_price(200)` `─×─ tax(1.1)` → `total(220)`

맨 오른쪽에서 `1`(= ∂total/∂total)로 시작해 **오른쪽 곱셈 노드 → 왼쪽 곱셈 노드** 순서로 손으로 계산하자.
- `d_apple_price`, `d_tax`: 오른쪽 곱셈 노드 (입력 `apple_price = 200`, `tax = 1.1`)
- `d_apple`, `d_apple_num`: 왼쪽 곱셈 노드 (입력 `apple = 100`, `apple_num = 2`, 상류는 `d_apple_price`)

In [ ]:
d_apple_price = None   # 여기에 작성
d_tax = None           # 여기에 작성
d_apple = None         # 여기에 작성
d_apple_num = None     # 여기에 작성

In [ ]:
# 확인
check(d_apple_price, 1.1, 'd_apple_price'); check(d_tax, 200, 'd_tax')
check(d_apple, 2.2, 'd_apple'); check(d_apple_num, 110, 'd_apple_num')
print('통과! d_apple = 2.2 — 1-4에서 수치 미분으로 구한 값과 같다.')

## 3-4. 문제: 사과와 귤 (그림 5-15 빈칸 채우기)
1-2의 그래프(사과 100×2, 귤 150×3, 더하기, ×1.1)를 **거꾸로** 계산하자. 시작 값은 `1`.
- `d_all_price`, `d_tax`: 맨 오른쪽 곱셈 노드 (입력 `all_price = 650`, `tax = 1.1`)
- `d_apple_price`, `d_orange_price`: 덧셈 노드
- `d_apple`, `d_apple_num`, `d_orange`, `d_orange_num`: 왼쪽 곱셈 노드 두 개

In [ ]:
d_all_price = None      # 여기에 작성
d_tax = None            # 여기에 작성
d_apple_price = None    # 여기에 작성
d_orange_price = None   # 여기에 작성
d_apple = None          # 여기에 작성
d_apple_num = None      # 여기에 작성
d_orange = None         # 여기에 작성
d_orange_num = None     # 여기에 작성

In [ ]:
# 확인
check(d_all_price, 1.1, 'd_all_price'); check(d_tax, 650, 'd_tax')
check(d_apple_price, 1.1, 'd_apple_price'); check(d_orange_price, 1.1, 'd_orange_price')
check(d_apple, 2.2, 'd_apple'); check(d_apple_num, 110, 'd_apple_num')
check(d_orange, 3.3, 'd_orange'); check(d_orange_num, 165, 'd_orange_num')
print('통과!')

---
# Part 4. 단순한 계층 구현하기 (5.4)
## 4-1. 곱셈 계층 `MulLayer` (5.4.1, 책 코드 `layer_naive.py`)
- `forward(x, y)`: 받은 `x`, `y`를 **`self.x`, `self.y`에 저장**하고 `x * y`를 돌려준다
- `backward(dout)`: `dx`, `dy`를 돌려준다 (3-2처럼 **바꿔서** 곱하기)

> **왜 `self.`인가?** `forward`의 `x`, `y`는 `forward`가 끝나면 사라지는 **지역 이름표**다.
> `backward`는 **다른 메서드**라서 그 이름표를 못 본다. 객체 자신(`self`)에 붙여 둬야 나중에 꺼낼 수 있다.
> (4장에서 정리한 "`self.`는 메서드 사이에서 값을 나눠 쓸 때 필요")

In [ ]:
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None

    def forward(self, x, y):
        pass   # 여기에 작성 (저장 2줄 + 계산 + return)

    def backward(self, dout):
        pass   # 여기에 작성 (dx, dy 계산 + return dx, dy)

In [ ]:
# 확인
m = MulLayer()
check(m.forward(10, 5), 50, 'forward(10, 5)')
check(m.backward(1.3), (6.5, 13.0), 'backward(1.3)')
print('통과!')

## 4-2. 사과 쇼핑을 계층으로 (책 코드 `buy_apple.py`)
계층(노드) 두 개를 만들어 두었다. **순전파 2줄, 역전파 2줄**을 채우자.
- 순전파: `apple_price`, `price`
- 역전파: 시작 값 `dprice = 1` → `dapple_price, dtax` → `dapple, dapple_num`

> ⚠️ 역전파는 순전파의 **반대 순서**로 부른다 (마지막 계층부터).

In [ ]:
apple, apple_num, tax = 100, 2, 1.1

mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()

# 순전파
apple_price = None   # 여기에 작성
price = None         # 여기에 작성

# 역전파
dprice = 1
dapple_price, dtax = None, None   # 여기에 작성
dapple, dapple_num = None, None   # 여기에 작성

print(price, dapple, dapple_num, dtax)

In [ ]:
# 확인
check(price, 220.0, 'price'); check(dapple, 2.2, 'dapple')
check(dapple_num, 110, 'dapple_num'); check(dtax, 200, 'dtax')
print('통과! 3-3에서 손으로 구한 값과 같다.')

## 4-3. 덧셈 계층 `AddLayer` (5.4.2)
### 🔮 먼저 생각하기
- `need_store`: 덧셈 노드의 역전파(3-1)에 순전파 때의 `x`, `y` 값이 **필요할까?** (`True` / `False`)

그다음 `AddLayer`를 만들자. (`__init__`은 할 일이 없어서 `pass`)

In [ ]:
need_store = None   # 여기에 작성

class AddLayer:
    def __init__(self):
        pass

    def forward(self, x, y):
        pass   # 여기에 작성

    def backward(self, dout):
        pass   # 여기에 작성 (return dx, dy)

In [ ]:
# 확인
check(need_store, False, 'need_store')
a = AddLayer()
check(a.forward(200, 450), 650, 'forward(200, 450)')
check(a.backward(1.1), (1.1, 1.1), 'backward(1.1)')
print('통과! 덧셈의 국소적 미분은 항상 1이라 x, y를 기억할 필요가 없다.')

## 4-4. 사과와 귤을 계층으로 (책 코드 `buy_apple_orange.py`)
계층 4개를 만들어 두었다. **순전파 4줄, 역전파 4줄**을 채우자. (3-4의 답과 같아야 한다)

In [ ]:
apple, apple_num = 100, 2
orange, orange_num = 150, 3
tax = 1.1

mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

# 순전파
apple_price = None    # 여기에 작성
orange_price = None   # 여기에 작성
all_price = None      # 여기에 작성
price = None          # 여기에 작성

# 역전파
dprice = 1
dall_price, dtax = None, None                # 여기에 작성
dapple_price, dorange_price = None, None     # 여기에 작성
dorange, dorange_num = None, None            # 여기에 작성
dapple, dapple_num = None, None              # 여기에 작성

print(price)
print(dapple_num, dapple, dorange, dorange_num, dtax)

In [ ]:
# 확인
check(price, 715.0, 'price')
check(dapple_num, 110, 'dapple_num'); check(dapple, 2.2, 'dapple')
check(dorange, 3.3, 'dorange'); check(dorange_num, 165, 'dorange_num'); check(dtax, 650, 'dtax')
print('통과!')

## 4-5. ⚠️🔮 계층 하나를 두 번 쓰면? (이름표와 상자)
아래 코드는 `MulLayer`를 **하나만** 만들어 사과와 귤에 둘 다 썼다. **실행하지 말고** 예측하자.
```python
mul = MulLayer()
apple_price = mul.forward(100, 2)     # 사과
orange_price = mul.forward(150, 3)    # 귤
dapple, dapple_num = mul.backward(1.0)
```
- `p_dapple`, `p_dapple_num`: 출력될 값 (원래 사과 쪽이라면 2, 100이 나와야 한다)

> 힌트: 두 번째 `forward`가 `self.x`, `self.y`를 어떻게 했을까?

In [ ]:
p_dapple = None       # 여기에 작성
p_dapple_num = None   # 여기에 작성

In [ ]:
# 확인 (이제 실행해서 맞춰 보기)
mul = MulLayer()
apple_price = mul.forward(100, 2)
orange_price = mul.forward(150, 3)
dapple, dapple_num = mul.backward(1.0)
print(dapple, dapple_num)
check(p_dapple, dapple, 'p_dapple'); check(p_dapple_num, dapple_num, 'p_dapple_num')
print('통과! self.x, self.y 상자에는 마지막 forward 값만 남는다 → 노드(계층) 하나 = 객체 하나.')

---
# Part 5. 활성화 함수 계층 구현하기 (5.5)
## 5-1. 🔮 `mask` 연습 (ReLU 준비)
ReLU: $x > 0$이면 $x$, 아니면 $0$. 역전파: $x > 0$이면 그대로 흘리고, 아니면 $0$.
책은 **True/False 배열(`mask`)**로 "0으로 만들 자리"를 기억한다. **실행하지 말고** 적자.
```python
x = np.array([[1.0, -0.5],
              [-2.0, 3.0]])
mask = (x <= 0)
out = x.copy()
out[mask] = 0
```
- `p_mask`: `mask`의 값 (True/False 2×2 리스트)
- `p_out`: `out`의 값 (2×2 리스트)

In [ ]:
p_mask = None   # 여기에 작성
p_out = None    # 여기에 작성

In [ ]:
# 확인
x = np.array([[1.0, -0.5], [-2.0, 3.0]])
mask = (x <= 0)
out = x.copy()
out[mask] = 0
print(mask); print(out)
check(np.array(p_mask).astype(int), mask.astype(int), 'p_mask (True=1, False=0)')
check(p_out, out, 'p_out')
print('통과! 배열[True/False 배열] = 값 → True인 자리만 바꾼다.')

## 5-2. ReLU 계층 (5.5.1)
- `forward(x)`: `self.mask`에 `(x <= 0)`을 저장, **`x.copy()`**한 `out`의 mask 자리를 0으로, `return out`
- `backward(dout)`: `dout`의 mask 자리를 0으로, 그걸 `dx`로 돌려주기

In [ ]:
class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        pass   # 여기에 작성

    def backward(self, dout):
        pass   # 여기에 작성

In [ ]:
# 확인
r = Relu()
x = np.array([[1.0, -0.5], [-2.0, 3.0]])
check(r.forward(x), np.array([[1.0, 0.0], [0.0, 3.0]]), 'forward')
check(x, np.array([[1.0, -0.5], [-2.0, 3.0]]), 'forward 후 x (원본 그대로?)')
check(r.backward(np.ones((2, 2))), np.array([[1.0, 0.0], [0.0, 1.0]]), 'backward')
print('통과!')

## 5-3. ⚠️🔮 `.copy()`를 빼면? (이름표와 상자)
```python
class ReluNoCopy:
    def forward(self, x):
        self.mask = (x <= 0)
        out = x            # .copy() 없음!
        out[self.mask] = 0
        return out

x = np.array([-1.0, 2.0, -3.0])
out = ReluNoCopy().forward(x)
print(x)
```
- `p_x_after`: 출력되는 `x` (리스트)

### 🔮 이어서 — 책의 `Relu.backward`는 `dout`을 고칠까?
```python
r2 = Relu()
r2.forward(np.array([-1.0, 2.0, -3.0]))
dout = np.array([5.0, 5.0, 5.0])
dx = r2.backward(dout)
print(dout, dx is dout)
```
- `p_dout_after`: 출력되는 `dout` (리스트)
- `p_same`: `dx is dout` (같은 상자인가? `True`/`False`)

In [ ]:
p_x_after = None      # 여기에 작성
p_dout_after = None   # 여기에 작성
p_same = None         # 여기에 작성

In [ ]:
# 확인 (이제 실행해서 맞춰 보기)
class ReluNoCopy:
    def forward(self, x):
        self.mask = (x <= 0)
        out = x
        out[self.mask] = 0
        return out

x = np.array([-1.0, 2.0, -3.0])
out = ReluNoCopy().forward(x)
print('x:', x)
check(p_x_after, x, 'p_x_after')

r2 = Relu()
r2.forward(np.array([-1.0, 2.0, -3.0]))
dout = np.array([5.0, 5.0, 5.0])
dx = r2.backward(dout)
print('dout:', dout, ' dx is dout:', dx is dout)
check(p_dout_after, dout, 'p_dout_after'); check(p_same, dx is dout, 'p_same')
print('통과!')
print('- out = x 는 새 상자가 아니라 이름표만 하나 더 → out을 고치면 x(입력 데이터)도 망가진다.')
print('- 책의 backward는 dout 상자를 직접 고친다. 역전파에서 dout은 다시 안 쓰니 문제없지만, 알아 두기.')

## 5-4. Sigmoid 계층 (5.5.2)
순전파 $y = \dfrac{1}{1 + \exp(-x)}$, 역전파 $\dfrac{\partial L}{\partial y}\, y(1-y)$
- `forward(x)`: `out`을 계산해 **`self.out`에 저장**하고 돌려주기
- `backward(dout)`: `dout * (1.0 - self.out) * self.out`

In [ ]:
class Sigmoid:
    def __init__(self):
        self.out = None

    def forward(self, x):
        pass   # 여기에 작성

    def backward(self, dout):
        pass   # 여기에 작성

In [ ]:
# 확인
s = Sigmoid()
check(s.forward(np.array([0.0])), np.array([0.5]), 'forward(0)')
check(s.backward(np.array([1.0])), np.array([0.25]), 'backward(1)')
s.forward(np.array([1.0]))
num = numerical_diff(lambda v: 1 / (1 + np.exp(-v)), 1.0)    # 4장 방식으로 검산
check(s.backward(np.array([1.0])), np.array([num]), 'x=1에서 역전파 = 수치 미분?')
print('통과!')

## 5-5. 생각해 보기 — 왜 `x`가 아니라 `out`을 저장할까?
- `p_need`: Sigmoid 역전파 식 $\dfrac{\partial L}{\partial y}\, y(1-y)$ 에 필요한 건 입력 `'x'`일까, 출력 `'y'`일까? (문자열)

In [ ]:
p_need = None   # 여기에 작성 ('x' 또는 'y')

In [ ]:
# 확인
check(p_need, 'y', 'p_need')
print('통과! 순전파 출력만 있으면 역전파가 된다 → exp를 다시 계산할 필요가 없다.')

---
# Part 6. Affine/Softmax 계층 구현하기 (5.6)
## 6-1. 🔁 Affine 계층 — 모양 예측 (5.6.1, 데이터 1개)
순전파 `Y = np.dot(X, W) + B`. `X` (2,), `W` (2, 3), `B` (3,)
역전파 식: $\dfrac{\partial L}{\partial X} = \dfrac{\partial L}{\partial Y} W^{T}$, $\dfrac{\partial L}{\partial W} = X^{T} \dfrac{\partial L}{\partial Y}$
- `p_Y`: `Y`의 모양
- `p_dX`: ∂L/∂X의 모양
- `p_dW`: ∂L/∂W의 모양

> 규칙: **기울기의 모양 = 원래 변수의 모양.** (4장 "dW 모양 = W 모양"과 같다)

In [ ]:
p_Y = None    # 여기에 작성 (튜플)
p_dX = None   # 여기에 작성
p_dW = None   # 여기에 작성

In [ ]:
# 확인
check(p_Y, (3,), 'p_Y'); check(p_dX, (2,), 'p_dX'); check(p_dW, (2, 3), 'p_dW')
print('통과!')

## 6-2. 🔁 배치용 Affine 계층 (5.6.2)
데이터 4개: `X` (4, 2), `W` (2, 3), 상류에서 온 `dY` (4, 3)
- `dX`: `np.dot(dY, W.T)`
- `dW`: `np.dot(X.T, dY)`
- `dB`: `dY`를 **데이터 방향(axis=0)으로 합친 것**

먼저 각 모양을 머릿속으로 예측해 보고(`(4,2)`? `(2,3)`? `(3,)`?), 그다음 세 줄을 쓰자.

In [ ]:
X = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])
W = np.array([[1.0, 0.0, -1.0], [0.0, 1.0, 1.0]])
dY = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 1.0]])

dX = None   # 여기에 작성
dW = None   # 여기에 작성
dB = None   # 여기에 작성

In [ ]:
# 확인
print(dX.shape, dW.shape, dB.shape)
check(dX, np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 1.0], [0.0, 2.0]]), 'dX')
check(dW, np.array([[8.0, 10.0, 12.0], [10.0, 12.0, 14.0]]), 'dW')
check(dB, np.array([2.0, 2.0, 2.0]), 'dB')
print('통과! dX = X 모양 (4, 2), dW = W 모양 (2, 3), dB = B 모양 (3,) — 데이터가 몇 개든 dW, dB 모양은 그대로.')

## 6-3. 🔁🔮 편향의 역전파는 왜 `axis=0` 합인가?
순전파에서 편향 `B` (3,)는 **브로드캐스트**로 모든 데이터(행)에 **똑같이** 더해졌다.
그래서 역전파에서는 데이터마다 온 기울기를 **다시 합쳐서** 편향 하나로 모은다. **실행하지 말고** 적자.
```python
dY = np.array([[1, 2, 3],
               [4, 5, 6]])
```
- `p_dB`: `np.sum(dY, axis=0)` (리스트)
- `p_wrong`: 잘못해서 `np.sum(dY, axis=1)`을 쓰면? (리스트)

In [ ]:
p_dB = None      # 여기에 작성
p_wrong = None   # 여기에 작성

In [ ]:
# 확인
dY = np.array([[1, 2, 3], [4, 5, 6]])
print(np.sum(dY, axis=0), np.sum(dY, axis=1))
check(p_dB, np.sum(dY, axis=0), 'p_dB'); check(p_wrong, np.sum(dY, axis=1), 'p_wrong')
print('통과! axis=1은 데이터마다 합 → 길이가 데이터 수(2)라서 편향 모양 (3,)과 안 맞는다.')

## 6-4. Affine 계층 클래스 (5.6.2)
- `__init__(W, b)`: `self.W`, `self.b`에 저장. `self.x`, `self.dW`, `self.db`는 `None`
- `forward(x)`: `self.x`에 저장, `np.dot(x, self.W) + self.b` 돌려주기
- `backward(dout)`: `dx`를 **돌려주고**, `dW`, `db`는 **`self.dW`, `self.db`에 저장**

> ⚠️ `dW`, `db`는 `return`하지 않는다. 다음 계층으로 흘려보낼 건 `dx`뿐이고,
> `dW`, `db`는 나중에 `TwoLayerNet`이 `self.layers['Affine1'].dW`처럼 **꺼내 간다**.

In [ ]:
class Affine:
    def __init__(self, W, b):
        pass   # 여기에 작성 (5줄)

    def forward(self, x):
        pass   # 여기에 작성

    def backward(self, dout):
        pass   # 여기에 작성

In [ ]:
# 확인 — 3·4장 남은 문제 6-1과 같은 숫자
Xs = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
Ws = np.array([[1.0, 0.0, -1.0], [0.0, 1.0, 1.0]])
bs = np.array([0.1, 0.2, 0.3])
aff = Affine(Ws, bs)
check(aff.forward(Xs), np.array([[1.1, 2.2, 1.3], [3.1, 4.2, 1.3], [5.1, 6.2, 1.3]]), 'forward')
dx = aff.backward(np.ones((3, 3)))
check(dx, np.array([[0.0, 2.0], [0.0, 2.0], [0.0, 2.0]]), 'dx')
check(aff.dW, np.array([[9.0, 9.0, 9.0], [12.0, 12.0, 12.0]]), 'self.dW')
check(aff.db, np.array([3.0, 3.0, 3.0]), 'self.db')
print('통과!')

## 6-5. Softmax-with-Loss 계층 (5.6.3)
출력층의 Softmax와 손실(교차 엔트로피)을 **한 계층**으로 묶는다.
- `forward(x, t)`: `self.t = t`, `self.y = softmax(x)`, `self.loss = cross_entropy_error(self.y, self.t)`, `return self.loss`
- `backward(dout=1)`: `(self.y - self.t) / batch_size` (`batch_size`는 `self.t.shape[0]`)

`t`는 **원-핫** (책 본문 버전).

In [ ]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None   # 손실
        self.y = None      # softmax의 출력
        self.t = None      # 정답 (원-핫)

    def forward(self, x, t):
        pass   # 여기에 작성

    def backward(self, dout=1):
        pass   # 여기에 작성

In [ ]:
# 확인 — 역전파 결과를 4장 수치 기울기로 검산 (lambda 주의: x_를 받아 손실을 돌려주는 함수)
x = np.array([[0.3, 2.9, 4.0], [1.0, 1.0, 1.0]])
t = np.array([[0, 0, 1], [1, 0, 0]])
layer = SoftmaxWithLoss()
check(layer.forward(x, t), cross_entropy_error(softmax(x), t), 'loss')
dx = layer.backward()
num = numerical_gradient(lambda x_: SoftmaxWithLoss().forward(x_, t), x.copy())
check(dx, num, 'backward = 수치 기울기?')
print('통과!')

## 6-6. 🔮 생각해 보기 — `(y - t)`가 뜻하는 것
데이터 1개(배치 크기 1): `y = [[0.3, 0.2, 0.5]]`, 정답 `t = [[0, 1, 0]]`
- `p_dx`: 역전파 결과 `(y - t) / 1` (2차원 리스트)
- `p_up`: 경사 하강법은 기울기의 **반대**로 움직인다. 그러면 **정답 자리(1번)**의 점수는 올라갈까? (`True`/`False`)

In [ ]:
p_dx = None   # 여기에 작성
p_up = None   # 여기에 작성

In [ ]:
# 확인
y = np.array([[0.3, 0.2, 0.5]]); t = np.array([[0, 1, 0]])
check(p_dx, (y - t) / 1, 'p_dx'); check(p_up, True, 'p_up')
print('통과! 역전파 값 = 예측과 정답의 차이(오차) 그대로. 정답 자리는 음수 → 빼면 올라가고, 오답 자리는 내려간다.')
print('batch_size로 나누는 이유: 순전파의 손실이 배치 평균이라서, 데이터 하나가 미치는 몫도 1/N.')

## 6-7. ⭐ 정답 레이블(숫자)도 받는 Softmax-with-Loss (책 `common/layers.py`, 본문 밖)
책의 `common/layers.py`는 `t`가 원-핫이든 **정답 숫자**든 받는다. 3·4장에서 연습한 **짝지어 꺼내기**를 쓴다.
- `t`가 정답 숫자 `(N,)`일 때: `dx = self.y.copy()` → `dx[np.arange(N), self.t] -= 1` → `dx = dx / N`

> ⚠️ 여기서도 `.copy()`가 필요하다. 없으면 `dx`와 `self.y`가 **같은 상자**라서 `self.y`가 망가진다.

In [ ]:
class SoftmaxWithLossLabel:
    def __init__(self):
        self.loss = None
        self.y = None
        self.t = None

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        if self.t.size == self.y.size:   # 원-핫이면
            dx = (self.y - self.t) / batch_size
        else:                            # 정답 숫자면
            pass   # 여기에 작성 (3줄)
        return dx

In [ ]:
# 확인 — 원-핫 버전(6-5)과 같은 결과?
x = np.array([[0.3, 2.9, 4.0], [1.0, 1.0, 1.0]])
a = SoftmaxWithLoss(); a.forward(x, np.array([[0, 0, 1], [1, 0, 0]]))
b = SoftmaxWithLossLabel(); b.forward(x, np.array([2, 0]))
y_before = b.y.copy()
check(b.backward(), a.backward(), '정답 숫자 버전 = 원-핫 버전?')
check(b.y, y_before, 'backward 후 self.y (그대로?)')
print('통과!')

---
# Part 7. 오차역전파법 구현하기 (5.7)
## 7-1. 신경망 학습의 전체 그림 (5.7.1)
4장과 같은 4단계: 1. 미니배치 → 2. **기울기 산출** → 3. 매개변수 갱신 → 4. 반복
- `step`: 오차역전파법이 빠르게 만들어 주는 단계 번호

In [ ]:
step = None   # 여기에 작성

In [ ]:
# 확인
check(step, 2, 'step')
print('통과! 나머지 단계는 4장과 똑같다.')

## 7-2. 🔮 `OrderedDict`로 계층 담기 (딕셔너리 연습)
`TwoLayerNet`은 계층을 **순서 있는 딕셔너리**에 담는다. **실행하지 말고** 적자.
```python
layers = OrderedDict()
layers['Affine1'] = 'A1'
layers['Relu1'] = 'R1'
layers['Affine2'] = 'A2'

p_keys     = list(layers.keys())
p_forward  = list(layers.values())

rev = list(layers.values())
ret = rev.reverse()
p_backward = rev
```
- `p_keys`, `p_forward`, `p_backward`: 리스트
- `p_ret_is_none`: `ret is None`인가? (`True`/`False`)

> 실제로는 `'A1'` 자리에 `Affine(...)` **객체**가 들어간다. 순전파는 넣은 순서대로, 역전파는 **거꾸로** 꺼낸다.

In [ ]:
p_keys = None          # 여기에 작성
p_forward = None       # 여기에 작성
p_backward = None      # 여기에 작성
p_ret_is_none = None   # 여기에 작성

In [ ]:
# 확인
layers = OrderedDict()
layers['Affine1'] = 'A1'; layers['Relu1'] = 'R1'; layers['Affine2'] = 'A2'
rev = list(layers.values()); ret = rev.reverse()
check(p_keys, list(layers.keys()), 'p_keys'); check(p_forward, list(layers.values()), 'p_forward')
check(p_backward, rev, 'p_backward'); check(p_ret_is_none, ret is None, 'p_ret_is_none')
print('통과! ⚠️ .reverse()는 리스트 자체를 뒤집고 None을 돌려준다 → rev = rev.reverse()라고 쓰면 rev가 None이 된다.')

## 7-3. 🔁 TwoLayerNet 모양 예측 (MNIST, 배치 100)
`TwoLayerNet(input_size=784, hidden_size=50, output_size=10)`, `x_batch` (100, 784)
- `p_a1`: `Affine1` 출력의 모양
- `p_r1`: `Relu1` 출력의 모양
- `p_a2`: `Affine2` 출력(= 점수)의 모양
- `p_dW1`: `layers['Affine1'].dW`의 모양
- `p_dx_last`: `lastLayer.backward()` 결과의 모양

In [ ]:
p_a1 = None        # 여기에 작성
p_r1 = None        # 여기에 작성
p_a2 = None        # 여기에 작성
p_dW1 = None       # 여기에 작성
p_dx_last = None   # 여기에 작성

In [ ]:
# 확인
check(p_a1, (100, 50), 'p_a1'); check(p_r1, (100, 50), 'p_r1'); check(p_a2, (100, 10), 'p_a2')
check(p_dW1, (784, 50), 'p_dW1'); check(p_dx_last, (100, 10), 'p_dx_last')
print('통과! ReLU는 모양을 안 바꾼다. 역전파 값의 모양 = 그 자리 순전파 값의 모양.')

## 7-4. 오차역전파법을 적용한 TwoLayerNet (5.7.2, 책 코드 `ch05/two_layer_net.py`)
`__init__`은 채워 두었다. 4개 메서드를 채우자. (책은 `common/layers.py`에서 계층을 import하지만, 여기서는 **위에서 직접 만든** `Affine`, `Relu`, `SoftmaxWithLoss`를 쓴다)
- `predict(x)`: `self.layers.values()`를 차례로 돌며 `x = layer.forward(x)` → `return x`
- `loss(x, t)`: `predict` 후 `self.lastLayer.forward(y, t)`
- `accuracy(x, t)`: 4장과 같음 (`t`가 원-핫이면 `argmax`)
- `numerical_gradient(x, t)`: **4장 그대로** — `loss_W = lambda W: self.loss(x, t)` 후 키마다 `numerical_gradient(loss_W, self.params[key])`
- `gradient(x, t)`: ① `self.loss(x, t)`로 순전파 ② `dout = self.lastLayer.backward(1)` ③ 계층을 **거꾸로** 돌며 `dout = layer.backward(dout)` ④ `grads` 딕셔너리에 `self.layers['Affine1'].dW` 등을 담아 돌려주기

> ⚠️ 메서드 `numerical_gradient` 안에서 부르는 `numerical_gradient(loss_W, ...)`는 **0번 준비 셀의 함수**다.
> 맨 이름은 클래스 메서드를 찾지 않는다(메서드는 `self.numerical_gradient`로만 부른다).

In [ ]:
class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 가중치 초기화 (4장과 같다)
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        # 계층 생성 — Affine에는 params 안의 배열(상자)을 그대로 넘긴다
        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])
        self.lastLayer = SoftmaxWithLoss()

    def predict(self, x):
        pass   # 여기에 작성

    def loss(self, x, t):
        pass   # 여기에 작성

    def accuracy(self, x, t):
        pass   # 여기에 작성

    def numerical_gradient(self, x, t):
        pass   # 여기에 작성

    def gradient(self, x, t):
        pass   # 여기에 작성

In [ ]:
# 확인 (작은 가짜 데이터)
np.random.seed(0)
net = TwoLayerNet(input_size=4, hidden_size=5, output_size=3)
x = np.random.rand(3, 4)
t = np.eye(3)                          # 원-핫 정답 3개
check(net.predict(x).shape, (3, 3), 'predict 모양')
check(np.ndim(net.loss(x, t)), 0, 'loss는 숫자 하나 (차원 0)')
check(net.accuracy(x, t) in (0.0, 1/3, 2/3, 1.0), True, 'accuracy 값이 0~1의 데이터 비율인가')
g = net.gradient(x, t)
check(sorted(g.keys()), ['W1', 'W2', 'b1', 'b2'], 'grads 키')
check(g['W1'].shape, net.params['W1'].shape, "grads['W1'] 모양 = W1 모양")
print('통과!')

## 7-5. 오차역전파법으로 구한 기울기 검증하기 (5.7.3, 책 코드 `gradient_check.py`)
수치 미분은 느리지만 구현이 단순해서 틀리기 어렵다 → **역전파 결과가 맞는지 검산**하는 데 쓴다(**기울기 확인**).
책은 MNIST 3장으로 하지만, 여기서는 빨리 끝나도록 작은 가짜 데이터로 한다.
- 키마다 `diff = np.average(np.abs(grad_backprop[key] - grad_numerical[key]))`를 구해 **딕셔너리 `diffs`**에 `diffs[key] = diff`로 담자. (딕셔너리 연습)

In [ ]:
np.random.seed(0)
net = TwoLayerNet(input_size=4, hidden_size=5, output_size=3)
x_batch = np.random.rand(3, 4)
t_batch = np.eye(3)

grad_numerical = net.numerical_gradient(x_batch, t_batch)
grad_backprop = net.gradient(x_batch, t_batch)

diffs = {}
# 여기에 작성 (for key in grad_numerical: ...)

print(diffs)

In [ ]:
# 확인
check(sorted(diffs.keys()), ['W1', 'W2', 'b1', 'b2'], 'diffs 키')
check(max(diffs.values()) < 1e-7, True, '차이가 모두 아주 작은가 (< 1e-7)')
print('통과! 오차가 0은 아니지만 매우 작다 → 역전파 구현이 맞다. (수치 미분의 반올림 오차 때문에 딱 0은 안 나온다)')

## 7-6. 오차역전파법을 사용한 학습 구현하기 (5.7.4, 책 코드 `ch05/train_neuralnet.py`)
4장 `train_neuralnet.py`와 **한 줄만** 다르다: `numerical_gradient` 대신 `gradient`.
- `grad`: 미니배치로 기울기 구하기 (역전파)
- 갱신: 키 4개를 돌며 `network.params[key] -= learning_rate * grad[key]`

> MNIST 6만 장으로 10,000번 반복한다. 1분 안팎 걸린다.

In [ ]:
from dataset.mnist import load_mnist

(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

np.random.seed(0)
network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    grad = None   # 여기에 작성 (오차역전파법으로 기울기)

    # 여기에 작성 (매개변수 갱신: for key in ('W1', 'b1', 'W2', 'b2'): ...)

    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)

    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print(train_acc, test_acc)

In [ ]:
# 확인
print('마지막 시험 정확도:', test_acc_list[-1])
check(test_acc_list[-1] > 0.95, True, '시험 정확도 > 0.95')
check(train_loss_list[-1] < train_loss_list[0], True, '손실이 줄었나')
print('통과! 4장(수치 미분)으로는 몇 시간 걸릴 학습이 1분 안팎에 끝났다.')

### 정확도 그래프 (실행만)

In [ ]:
x_axis = np.arange(len(train_acc_list))
plt.plot(x_axis, train_acc_list, marker='o', label='train acc')
plt.plot(x_axis, test_acc_list, marker='s', linestyle='--', label='test acc')
plt.xlabel('epochs'); plt.ylabel('accuracy'); plt.ylim(0, 1.0); plt.legend(loc='lower right')
plt.show()

## 7-7. ⚠️🔮 `-=` 대신 `= ... -`로 갱신하면? (이름표와 상자)
`__init__`에서 `Affine(self.params['W1'], ...)`은 **같은 상자**를 넘겼다. 그래서 `params['W1'] -= ...`로 상자 안 숫자를 고치면 `Affine1`도 바뀐 값을 쓴다.
그런데 아래처럼 쓰면? **실행하지 말고** 예측하자.
```python
np.random.seed(0)
net = TwoLayerNet(4, 5, 3)
x = np.random.rand(3, 4); t = np.eye(3)

before = net.loss(x, t)
grads = net.gradient(x, t)
for key in ('W1', 'b1', 'W2', 'b2'):
    net.params[key] = net.params[key] - 1.0 * grads[key]   # -= 대신
after = net.loss(x, t)
```
- `p_changed`: 손실이 바뀌었을까? (`after != before`, `True`/`False`)
- `p_same_box`: `net.params['W1'] is net.layers['Affine1'].W` 인가? (`True`/`False`)

In [ ]:
p_changed = None    # 여기에 작성
p_same_box = None   # 여기에 작성

In [ ]:
# 확인 (이제 실행해서 맞춰 보기)
np.random.seed(0)
net = TwoLayerNet(4, 5, 3)
x = np.random.rand(3, 4); t = np.eye(3)
before = net.loss(x, t)
grads = net.gradient(x, t)
for key in ('W1', 'b1', 'W2', 'b2'):
    net.params[key] = net.params[key] - 1.0 * grads[key]
after = net.loss(x, t)
print(before, after)
check(p_changed, bool(after != before), 'p_changed')
check(p_same_box, net.params['W1'] is net.layers['Affine1'].W, 'p_same_box')
print('통과! params의 이름표만 새 상자로 옮겨졌고, 계층들은 여전히 옛 상자를 본다 → 학습이 안 된다.')
print('책이 갱신을 -= 로 쓰는 이유. (3·4장 남은 문제 E-9와 같은 이야기)')

## 7-8. 수치 미분 vs 오차역전파법 — 속도 비교 (실행만)
같은 신경망, 같은 데이터로 기울기를 한 번 구하는 시간을 재 보자.

In [ ]:
import time
np.random.seed(0)
net = TwoLayerNet(input_size=100, hidden_size=50, output_size=10)
x = np.random.rand(10, 100); t = np.eye(10)

start = time.time(); net.numerical_gradient(x, t); t_num = time.time() - start
start = time.time(); net.gradient(x, t); t_bp = time.time() - start
print(f'수치 미분: {t_num:.3f}초,  오차역전파법: {t_bp:.5f}초,  약 {t_num / max(t_bp, 1e-6):.0f}배')
print('가중치가 5,560개 → 수치 미분은 손실을 11,000번 넘게 계산한다. 역전파는 순전파 1번 + 역전파 1번.')

---
# 끝!
5장을 다 풀었으면 `ch05_review.ipynb`(복습)로 넘어가자. 오늘 막힌 곳은 노트(`5장 오차역전파법.md`)에 정리.